In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Importing Libs

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
from PIL import Image
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Utilizing GPU

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


## Dataset Class

In [ ]:
class AnimalDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

        # get unique labels and create mapping
        self.labels = sorted(self.data['label'].unique())
        self.label_to_idx = {label: idx for idx, label in enumerate(self.labels)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label']

        # load image
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label_idx = self.label_to_idx[label]
        return image, label_idx

## Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Evalutaion Function

In [ ]:
def evaluate_model(model, data_loader, return_predictions=True):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            # InceptionV3 returns an InceptionOutputs object, need to access .logits
            if isinstance(outputs, torch.nn.modules.container.Sequential) or not hasattr(outputs, 'logits'):
                # If it's a Sequential model or doesn't have logits (e.g., after custom head), use outputs directly
                _, predicted = torch.max(outputs.data, 1)
            else:
                # For InceptionV3 and similar models, use outputs.logits
                _, predicted = torch.max(outputs.logits.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    if return_predictions:
        return all_preds, all_labels
    else:
        acc = accuracy_score(all_labels, all_preds)
        return acc * 100

## Training Function

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20):
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            # Use outputs.logits for the main output in InceptionV3
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            pbar.set_postfix({'loss': running_loss/len(train_loader), 'acc': 100*correct/total})

        train_acc = 100 * correct / total

        # validation
        val_acc = evaluate_model(model, val_loader, return_predictions=False)
        print(f'Epoch {epoch+1}: Train Acc = {train_acc:.2f}%, Val Acc = {val_acc:.2f}%')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')

    model.load_state_dict(torch.load('best_model.pth'))
    return model

## Calculate Metrics

In [ ]:
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    return {
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

## Training

In [ ]:
from torchvision.models import Inception_V3_Weights

base_path = '/content/drive/Shareddrives/STAI_Project/datasets/csv/splits_csv'
num_folds = 1

results = {
    'incipitonv3': [],
}

for fold in range(1, num_folds + 1):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}/{num_folds}')
    print(f'{"="*60}')

    fold_path = os.path.join(base_path, f'fold_{fold}')

    train_csv = os.path.join(fold_path, 'train.csv')
    val_csv   = os.path.join(fold_path, 'val.csv')
    test_csv  = os.path.join(fold_path, 'test.csv')

    # ===============================
    # Datasets
    # ===============================
    train_dataset = AnimalDataset(train_csv, transform=train_transform)
    val_dataset   = AnimalDataset(val_csv,   transform=test_transform)
    test_dataset  = AnimalDataset(test_csv,  transform=test_transform)

    num_classes = len(train_dataset.labels)
    print(f'Number of classes: {num_classes}')
    print(f'Classes: {train_dataset.labels}')

    # ===============================
    # DataLoaders
    # ===============================
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

    # ===============================
    # InceptionV3 – Partial Fine-Tuning
    # ===============================
    print(f'\nTraining InceptionV3 (Partial Fine-Tuning) on Fold {fold}...')

    incipition_model = models.inception_v3(
        weights=Inception_V3_Weights.DEFAULT
    )

    # Replace FC
    num_ftrs = incipition_model.fc.in_features
    incipition_model.fc = nn.Linear(num_ftrs, num_classes)

    incipition_model = incipition_model.to(device)

    # ===============================
    # 🔒 Freeze all layers
    # ===============================
    for param in incipition_model.parameters():
        param.requires_grad = False

    # ===============================
    # 🔓 Unfreeze last block + FC
    # ===============================
    for param in incipition_model.Mixed_7c.parameters():
        param.requires_grad = True

    for param in incipition_model.fc.parameters():
        param.requires_grad = True

    # ===============================
    # Loss & Optimizer
    # ===============================
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, incipition_model.parameters()),
        lr=1e-4
    )

    # ===============================
    # Training
    # ===============================
    incipition_model = train_model(
        incipition_model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        epochs=5
    )

    # ===============================
    # Evaluation
    # ===============================
    preds, labels = evaluate_model(incipition_model, test_loader)
    incipition_metrics = calculate_metrics(labels, preds)
    results['incipitonv3'].append(incipition_metrics)

    print(f'\nInceptionV3 Results (Fold {fold}):')
    for metric, value in incipition_metrics.items():
        print(f'{metric}: {value:.4f}')



FOLD 1/1
Number of classes: 64
Classes: ['antelope', 'bear', 'beaver', 'bee', 'bison', 'blackbird', 'buffalo', 'butterfly', 'camel', 'cat', 'cheetah', 'chimpanzee', 'chinchilla', 'cow', 'crab', 'crocodile', 'deer', 'dog', 'dolphin', 'donkey', 'duck', 'eagle', 'elephant', 'falcon', 'ferret', 'flamingo', 'fox', 'frog', 'giraffe', 'goat', 'goose', 'gorilla', 'grasshopper', 'hawk', 'hedgehog', 'hippopotamus', 'hyena', 'iguana', 'jaguar', 'kangaroo', 'koala', 'lemur', 'leopard', 'lizard', 'lynx', 'mole', 'mongoose', 'ostrich', 'otter', 'owl', 'panda', 'peacock', 'penguin', 'porcupine', 'raccoon', 'seal', 'sheep', 'snail', 'snake', 'spider', 'squid', 'walrus', 'whale', 'wolf']

Training InceptionV3 (Partial Fine-Tuning) on Fold 1...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 111MB/s]
Epoch 1/5: 100%|██████████| 288/288 [1:29:36<00:00, 18.67s/it, loss=0.994, acc=88.9]


Epoch 1: Train Acc = 88.88%, Val Acc = 98.26%


Epoch 2/5: 100%|██████████| 288/288 [03:16<00:00,  1.47it/s, loss=0.093, acc=98.8]


Epoch 2: Train Acc = 98.84%, Val Acc = 99.39%


...

Epoch 3: Train Acc = 99.50%, Val Acc = 99.61%


Epoch 4/5: 100%|██████████| 288/288 [03:09<00:00,  1.52it/s, loss=0.0238, acc=99.7]


Epoch 4: Train Acc = 99.74%, Val Acc = 99.65%


Epoch 5/5: 100%|██████████| 288/288 [03:08<00:00,  1.53it/s, loss=0.0171, acc=99.8]


Epoch 5: Train Acc = 99.77%, Val Acc = 99.65%

InceptionV3 Results (Fold 1):
Accuracy: 0.9965
Precision: 0.9967
Recall: 0.9965
F1-Score: 0.9965
